In [1]:
import os

BASE_URL = f"http://localhost:8000/v1"

os.environ["BASE_URL"]    = BASE_URL
os.environ["OPENAI_API_KEY"] = "abc-123"   

print("Config set:", BASE_URL)

Config set: http://localhost:8000/v1


In [2]:
!pip install pydantic-ai-slim openai pandas numpy



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [3]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

provider = OpenAIProvider(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

llm_30b = OpenAIChatModel("Qwen3-30B-A3B", provider=provider)

In [4]:
!pip install pandas numpy faker



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [5]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()
np.random.seed(42)
random.seed(42)

# -----------------------------
# CONFIG
# -----------------------------
NUM_USERS = 10000
MAX_DEVICES_PER_USER = 3
NUM_EVENTS = 500000

# -----------------------------
# 1. USER PROFILE DATASET
# -----------------------------
def generate_users(n):
    users = []
    for i in range(n):
        user_id = f"U{i}"
        users.append({
            "user_id": user_id,
            "account_age_days": random.randint(30, 2000),
            "avg_login_per_day": round(random.uniform(1, 6), 2),
            "risk_tolerance_score": round(random.uniform(0, 1), 2)
        })
    return pd.DataFrame(users)

# -----------------------------
# 2. DEVICE + SIM DATASET
# -----------------------------
def generate_devices(users_df):
    devices = []
    for _, row in users_df.iterrows():
        user_id = row["user_id"]
        num_devices = random.randint(1, MAX_DEVICES_PER_USER)

        for d in range(num_devices):
            device_id = f"D_{user_id}_{d}"
            devices.append({
                "device_id": device_id,
                "user_id": user_id,
                "imei": fake.uuid4(),
                "imsi": fake.uuid4(),
                "device_type": random.choice(["android", "ios"]),
                "os_version": random.choice(["13", "14", "15"]),
                "is_primary": d == 0,
                "sim_change_count": random.randint(0, 3),
                "last_sim_change_days": random.randint(0, 365)
            })
    return pd.DataFrame(devices)

# -----------------------------
# 3. BEHAVIOR DATASET
# -----------------------------
def generate_behavior(users_df):
    behavior = []
    for _, row in users_df.iterrows():
        behavior.append({
            "user_id": row["user_id"],
            "avg_session_duration": round(random.uniform(5, 60), 2),
            "login_time_variance": round(random.uniform(0, 5), 2),
            "touch_pattern_score": round(random.uniform(0.7, 1.0), 2),
            "location_stability_score": round(random.uniform(0.6, 1.0), 2),
            "device_switch_frequency": round(random.uniform(0, 0.5), 2)
        })
    return pd.DataFrame(behavior)

# -----------------------------
# 4. LOGIN EVENTS DATASET
# -----------------------------
def generate_login_events(users_df, devices_df, n_events):
    events = []

    device_map = devices_df.groupby("user_id")["device_id"].apply(list).to_dict()

    for i in range(n_events):
        user = random.choice(users_df["user_id"].tolist())
        device = random.choice(device_map[user])

        timestamp = datetime.now() - timedelta(minutes=random.randint(0, 100000))

        events.append({
            "session_id": f"S{i}",
            "user_id": user,
            "device_id": device,
            "timestamp": timestamp,
            "login_hour": timestamp.hour,
            "network_type": random.choice(["4G", "5G", "WiFi"]),
            "cell_id": f"CELL_{random.randint(1,1000)}",
            "geo_distance_from_last_login": round(random.uniform(0, 50), 2),
            "is_roaming": random.choice([True, False]),
            "is_new_device": random.choice([True, False]),
            "sim_changed_recently": random.choice([True, False])
        })

    return pd.DataFrame(events)

# -----------------------------
# 5. RISK LABEL DATASET
# -----------------------------
def generate_labels(events_df):
    labels = []

    for _, row in events_df.iterrows():
        risk_score = 0

        # Simple heuristic
        if row["is_new_device"]:
            risk_score += 0.4
        if row["sim_changed_recently"]:
            risk_score += 0.5
        if row["geo_distance_from_last_login"] > 20:
            risk_score += 0.3

        risk_score = min(risk_score, 1.0)

        risk_level = "low"
        if risk_score > 0.7:
            risk_level = "high"
        elif risk_score > 0.3:
            risk_level = "medium"

        labels.append({
            "session_id": row["session_id"],
            "risk_score": risk_score,
            "risk_level": risk_level,
            "otp_triggered": risk_level != "low",
            "is_fraud": 1 if risk_level == "high" else 0
        })

    return pd.DataFrame(labels)

# -----------------------------
# RUN PIPELINE
# -----------------------------
users_df = generate_users(NUM_USERS)
devices_df = generate_devices(users_df)
behavior_df = generate_behavior(users_df)
events_df = generate_login_events(users_df, devices_df, NUM_EVENTS)
labels_df = generate_labels(events_df)

# Save
users_df.to_csv("users.csv", index=False)
devices_df.to_csv("devices.csv", index=False)
behavior_df.to_csv("behavior.csv", index=False)
events_df.to_csv("events.csv", index=False)
labels_df.to_csv("labels.csv", index=False)

print("Datasets generated successfully!")

Datasets generated successfully!


In [7]:
!pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 MB 86.4 MB/s  0:00:04:00:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyspark: filename=pyspark-4.1.2-py2.py3-none-any.whl size=456079515 sha256=e2788f6e7ba894cd4e8697926bf08da38abeeb157e4f41a7a3a430f537668607
  Stored in directory: /root/.cache/pip/wheels/e6/9c/35/b08622081a09dc48b9467b570ae170519430915aa3c8d27cf9
Successfully built pyspark
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyspark]m1/2 [pyspark]

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [8]:
from pyspark.sql.types import *

user_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("account_age_days", IntegerType(), True),
    StructField("avg_login_per_day", DoubleType(), True),
    StructField("risk_tolerance_score", DoubleType(), True)
])

In [9]:
device_schema = StructType([
    StructField("device_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("imei", StringType(), True),
    StructField("imsi", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("os_version", StringType(), True),
    StructField("is_primary", BooleanType(), True),
    StructField("sim_change_count", IntegerType(), True),
    StructField("last_sim_change_days", IntegerType(), True)
])

In [10]:
behavior_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("avg_session_duration", DoubleType(), True),
    StructField("login_time_variance", DoubleType(), True),
    StructField("touch_pattern_score", DoubleType(), True),
    StructField("location_stability_score", DoubleType(), True),
    StructField("device_switch_frequency", DoubleType(), True)
])

In [11]:
events_schema = StructType([
    StructField("session_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("login_hour", IntegerType(), True),
    StructField("network_type", StringType(), True),
    StructField("cell_id", StringType(), True),
    StructField("geo_distance_from_last_login", DoubleType(), True),
    StructField("is_roaming", BooleanType(), True),
    StructField("is_new_device", BooleanType(), True),
    StructField("sim_changed_recently", BooleanType(), True)
])

In [12]:
labels_schema = StructType([
    StructField("session_id", StringType(), True),
    StructField("risk_score", DoubleType(), True),
    StructField("risk_level", StringType(), True),
    StructField("otp_triggered", BooleanType(), True),
    StructField("is_fraud", IntegerType(), True)
])

In [13]:
from pydantic_ai import Tool
import random

@Tool
def context_agent(user_id: str, device_id: str) -> dict:
    """Returns device consistency and new device detection"""
    
    device_consistency = round(random.uniform(0.7, 1.0), 2)
    is_new_device = random.choice([True, False])
    
    return {
        "device_consistency": device_consistency,
        "is_new_device": is_new_device
    }

In [14]:
@Tool
def telecom_agent(user_id: str) -> dict:
    """Returns SIM and telecom-related signals"""
    
    sim_consistency = round(random.uniform(0.7, 1.0), 2)
    sim_changed_recently = random.choice([True, False])
    
    return {
        "sim_consistency": sim_consistency,
        "sim_changed_recently": sim_changed_recently
    }

In [15]:
@Tool
def behavior_agent(user_id: str) -> dict:
    """Returns behavioral consistency score"""
    
    behavior_score = round(random.uniform(0.6, 1.0), 2)
    
    return {
        "behavior_score": behavior_score
    }

In [16]:
def risk_agent(features: dict) -> dict:
    """
    ML-based risk scoring (simulated)
    """
    
    risk = (
        0.4 * (1 - features["device_consistency"]) +
        0.4 * (1 - features["sim_consistency"]) +
        0.2 * (1 - features["behavior_score"])
    )
    
    risk = round(min(risk, 1.0), 2)
    
    if risk > 0.7:
        level = "high"
    elif risk > 0.3:
        level = "medium"
    else:
        level = "low"
    
    return {
        "risk_score": risk,
        "risk_level": level
    }

In [17]:
from pydantic_ai import Agent

decision_agent = Agent(
    model=llm_30b,
    system_prompt="""
    You are a security decision engine.

    Input: risk_score, signals

    Rules:
    - risk < 0.3 → No OTP
    - 0.3–0.6 → Step-up auth
    - >0.6 → OTP required

    Output JSON:
    {
      "decision": "...",
      "reason": "..."
    }
    """
)

In [18]:
import asyncio

async def run_auth_flow(user_id: str, device_id: str):
    
    # Step 1: Collect signals
    ctx = context_agent(user_id=user_id, device_id=device_id)
    tel = telecom_agent(user_id=user_id)
    beh = behavior_agent(user_id=user_id)
    
    # Step 2: Merge features
    features = {
        **ctx,
        **tel,
        **beh
    }
    
    print("🔍 Features:", features)
    
    # Step 3: Risk scoring (ML)
    risk = risk_agent(features)
    
    print("⚠️ Risk:", risk)
    
    # Step 4: Decision (LLM)
    prompt = f"""
    Risk Score: {risk['risk_score']}
    Risk Level: {risk['risk_level']}
    Features: {features}
    """
    
    result = await decision_agent.run(prompt)
    
    return {
        "features": features,
        "risk": risk,
        "decision": result.output
    }

In [20]:
agent = Agent(
    model=llm_30b,
    tools=[context_agent, telecom_agent, behavior_agent]
)

result = await agent.run("""
Get device, telecom, and behavior signals for:
user_id=U123, device_id=D001
""")

print("\n✅ FINAL OUTPUT:\n", result)


✅ FINAL OUTPUT:
 AgentRunResult(output='\n\nHere are the signals for user U123 and device D001:\n\n**Device Signals**\n- Device consistency: 0.88 (high consistency)\n- Is new device: false\n\n**Telecom Signals**\n- SIM consistency: 1.0 (perfect consistency)\n- SIM changed recently: false\n\n**Behavior Signals**\n- Behavioral consistency score: 0.61 (moderate consistency)\n\nThe combined risk assessment would be moderate, primarily due to the lower behavioral consistency score. All device and telecom signals indicate normal activity with no recent changes.')


In [21]:
agent = Agent(
    model=llm_30b,
    tools=[context_agent, telecom_agent, behavior_agent],
    system_prompt="""
    You are a signal aggregation agent.

    You MUST:
    1. Call all tools (context_agent, telecom_agent, behavior_agent)
    2. Combine outputs
    3. Return ONLY JSON in this exact format:

    {
      "device_consistency": float,
      "is_new_device": bool,
      "sim_consistency": float,
      "sim_changed_recently": bool,
      "behavior_score": float
    }

    DO NOT return explanations.
    DO NOT return text.
    ONLY JSON.
    """
)

In [22]:
result = await agent.run("""
Fetch signals for:
user_id=U123
device_id=D001
Return JSON only
""")

In [23]:
import json

features = json.loads(result.output)

print("✅ Structured Features:", features)

✅ Structured Features: {'device_consistency': 0.93, 'is_new_device': True, 'sim_consistency': 0.81, 'sim_changed_recently': True, 'behavior_score': 0.71}


In [29]:
def risk_agent(features):
    risk = llm_30b.predict(features)

    # HARD GUARDRAILS
    if features["sim_changed_recently"]:
        return 1.0   # force high risk

    if features["is_new_device"] and features["behavior_score"] < 0.5:
        return max(risk, 0.8)

    return risk

In [28]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

provider = OpenAIProvider(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

agent_model = OpenAIChatModel("Qwen3-30B-A3B", provider=provider)

In [30]:
risk = risk_agent(features)

decision = await decision_agent.run(f"""
Risk Score: {risk['risk_score']}
Risk Level: {risk['risk_level']}
Features: {features}
""")

AttributeError: 'OpenAIChatModel' object has no attribute 'predict'